<h1><font color="#113D68" size=6>Deep Learning en Python con PyTorch</font></h1>

<h1><font color="#113D68" size=5>Parte 3. MultiLayer Perceptron</font></h1>

<h1><font color="#113D68" size=4>3. Desarrollar una red neuronal</font></h1>

<br><br>
<div style="text-align: right">
<font color="#113D68" size=3>Manuel Castillo Cara</font><br>

</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
More information about [Manuel Castillo-Cara](https://www.manuelcastillo.eu/)

<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i>
Puedes ver más cursos de Inteligencia Artificial, Machine Learning y Deep Learning en mi [página web](https://www.manuelcastillo.eu/udemy/)


---

<a id="indice"></a>
<h2><font color="#004D7F" size=5>Índice</font></h2>

* [0. Contexto](#section0)
* [1. Conjunto de datos](#section1)
* [2. Cargar datos](#section2)
* [3. Definir modelo](#section3)
* [4. Compilar modelo](#section4)
* [5. Ajustar modelo](#section5)
* [6. Evaluar modelo](#section6)
* [7. Ejercicio completo](#section7)
* [8. Hacer predicciones](#section8)


---
<a id="section0"></a>
# <font color="#004D7F" size=6> 0. Contexto</font>

PyTorch es una potente biblioteca de Python para construir modelos de aprendizaje profundo. Proporciona todo lo que necesitas para definir y entrenar una red neuronal y utilizarla para la inferencia. No necesitas escribir mucho código para completar todo esto. En este capítulo, descubrirás cómo crear tu primer modelo de red neuronal profunda en Python usando PyTorch. Después de completar este capítulo, sabrás:

- Cómo cargar un conjunto de datos CSV y prepararlo para usar con PyTorch
- Cómo definir un modelo Perceptrón Multicapa en PyTorch
- Cómo entrenar y evaluar un modelo PyTorch en un conjunto de datos de validación

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
[PyTorch tutorials.](https://pytorch.org/tutorials/)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
[PyTorch Documentation](https://pytorch.org/docs/stable/index.html).

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
PyTorch Tutorials. [Build the Neural Network](https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html).

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
PyTorch Tutorials. [Optimizing Model Parameters](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html).

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section1"></a>
# <font color="#004D7F" size=6>1. Cargar Datos</font>

El primer paso es definir las funciones y clases que usarás en este capítulo. Utilizarás la biblioteca NumPy para cargar tu conjunto de datos y la biblioteca PyTorch para los modelos de aprendizaje profundo.

Los imports requeridos son los siguientes:

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# Descarga de datos (necesario en Colab): se guardan en ./Datasets
import os, urllib.request

os.makedirs('Datasets', exist_ok=True)
urls = {
    'pima-indians-diabetes.csv':
        'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv',
}
for nombre, url in urls.items():
    if not os.path.exists(f'Datasets/{nombre}'):
        urllib.request.urlretrieve(url, f'Datasets/{nombre}')

En este capítulo, utilizarás el conjunto de datos de diabetes. 

- Describe los datos de registros médicos de los Indios Pima y si presentaron un inicio de diabetes en un plazo de cinco años. 

- Es un problema de clasificación binaria (inicio de diabetes como 1 o no como 0). 

- Todas las variables de entrada que describen a cada paciente están transformadas y son numéricas. 

Hay ocho variables de entrada y una variable de salida (la última columna).

**Variables de Entrada (X):**
1. Número de veces embarazada  

2. Concentración de glucosa en plasma a las 2 horas en una prueba de tolerancia oral a la glucosa  

3. Presión arterial diastólica (mm Hg)  

4. Espesor del pliegue cutáneo del tríceps (mm)  

5. Insulina sérica a las 2 horas (μIU/ml)  

6. Índice de masa corporal (peso en kg/(altura en m)^2)  

7. Función de pedigrí de diabetes  

8. Edad (años)

**Variable de Salida (y):**
- Etiqueta de clase (0 o 1)

Cargamos el archivo como una matriz de números usando la función `loadtxt()` de NumPy. 

In [3]:
# cargar el conjunto de datos, dividir en variables de entrada (X) y salida (y)
dataset = np.loadtxt('Datasets/pima-indians-diabetes.csv', delimiter=',') #Datasets/pima-indians-diabetes.csv
X = dataset[:, 0:8]
y = dataset[:, 8]
print(X)
print(y[0:10])

[[  6.    148.     72.    ...  33.6     0.627  50.   ]
 [  1.     85.     66.    ...  26.6     0.351  31.   ]
 [  8.    183.     64.    ...  23.3     0.672  32.   ]
 ...
 [  5.    121.     72.    ...  26.2     0.245  30.   ]
 [  1.    126.     60.    ...  30.1     0.349  47.   ]
 [  1.     93.     70.    ...  30.4     0.315  23.   ]]
[1. 0. 1. 0. 1. 0. 1. 0. 1. 1.]


Sin embargo, es necesario transformar estos datos en tensores de PyTorch previamente. 
- Uno de los motivos es que PyTorch generalmente opera con números de punto flotante de 32 bits, mientras que NumPy, por defecto, utiliza punto flotante de 64 bits.

- La combinación de estos tipos no está permitida en la mayoría de las operaciones, por tanto, se debe de convertir.

- Debemos utilizar `reshape(-1,1)` para reorganizar `y` en una matriz de una columna, donde -1 indica que PyTorch debe inferir automáticamente la dimensión correcta.


Para realizar la conversión, se crea un tensor a partir de matrices de NumPy de la siguiente manera:

In [4]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

print(X)
print(y[0:10])

tensor([[  6.0000, 148.0000,  72.0000,  ...,  33.6000,   0.6270,  50.0000],
        [  1.0000,  85.0000,  66.0000,  ...,  26.6000,   0.3510,  31.0000],
        [  8.0000, 183.0000,  64.0000,  ...,  23.3000,   0.6720,  32.0000],
        ...,
        [  5.0000, 121.0000,  72.0000,  ...,  26.2000,   0.2450,  30.0000],
        [  1.0000, 126.0000,  60.0000,  ...,  30.1000,   0.3490,  47.0000],
        [  1.0000,  93.0000,  70.0000,  ...,  30.4000,   0.3150,  23.0000]])
tensor([[1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.]])


Como se observa aparece un `warning`, podemos hacer caso y utilizar lo que se recomienda.

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section2"></a>
# <font color="#004D7F" size=6>2. Definir el modelo</font>

Hay dos formas principales de definir un modelo en PyTorch. Los puntos clave son:

1. Un modelo puede definirse como una secuencia de capas usando `Sequential`.

2. La primera capa debe tener el número correcto de características de entrada.

3. El diseño del modelo (número de capas, neuronas) a menudo se determina por experimentación.

4. Se utilizan capas totalmente conectadas (`Linear`) en este ejemplo.

5. Se emplean funciones de activación después de cada capa: ReLU para las capas ocultas y sigmoide para la capa de salida ya que es un problema de clasificación binaria.

6. La estructura del modelo propuesto es:

   - Entrada: 8 variables

   - Primera capa oculta: 12 neuronas con ReLU

   - Segunda capa oculta: 8 neuronas con ReLU
   
   - Capa de salida: 1 neurona con sigmoide

Este enfoque permite crear un modelo de red neuronal para clasificación binaria, con una salida entre 0 y 1.

In [5]:
model = nn.Sequential(
    nn.Linear(8, 12),
    nn.ReLU(),
    nn.Linear(12, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)
print(model)

Sequential(
  (0): Linear(in_features=8, out_features=12, bias=True)
  (1): ReLU()
  (2): Linear(in_features=12, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=1, bias=True)
  (5): Sigmoid()
)


Sin embargo, debes tener en cuenta que en PyTorch hay una forma más detallada de crear un modelo. 

- El modelo mencionado anteriormente se puede crear como una clase de Python que hereda de `nn.Module`.

- La clase debe definir todas las capas en el constructor (`__init__`).

- Es necesario preparar todos los componentes al crear la clase, aunque aún no se haya proporcionado la entrada.

- Se debe llamar al constructor de la clase principal (`super().__init__()`) para inicializar correctamente el modelo.

- Es obligatorio definir una función `forward()` dentro de la clase.

- La función `forward()` especifica cómo se genera el tensor de salida a partir de un tensor de entrada `x`.

- El modelo recordará los nombres asignados a cada capa, como se puede observar en la salida.

In [6]:
class PimaClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = nn.Linear(8, 12)
        self.act1 = nn.ReLU()
        self.hidden2 = nn.Linear(12, 8)
        self.act2 = nn.ReLU()
        self.output = nn.Linear(8, 1)
        self.act_output = nn.Sigmoid()

    def forward(self, x):
        x = self.act1(self.hidden1(x))
        x = self.act2(self.hidden2(x))
        x = self.act_output(self.output(x))
        return x

model = PimaClassifier()
print(model)

PimaClassifier(
  (hidden1): Linear(in_features=8, out_features=12, bias=True)
  (act1): ReLU()
  (hidden2): Linear(in_features=12, out_features=8, bias=True)
  (act2): ReLU()
  (output): Linear(in_features=8, out_features=1, bias=True)
  (act_output): Sigmoid()
)


---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section3"></a>
# <font color="#004D7F" size=6>3. Preparación para el entrenamiento</font>

- El modelo definido está listo para entrenarse, pero primero se debe especificar el objetivo del entrenamiento.

- Los datos tienen características de entrada `X` y una etiqueta de salida `y`.

- El objetivo es que el modelo produzca una salida lo más cercana posible a `y`.

- Entrenar la red implica encontrar el mejor conjunto de pesos para mapear entradas a salidas en el conjunto de datos.

- La **función de pérdida** mide qué tan cerca está la predicción de `y`.

- Para un problema de clasificación binaria, se utiliza la `binary cross-entropy`.

- Se necesita un **optimizador** para ajustar progresivamente los pesos del modelo.

- En este ejemplo se usa `Adam`, una versión popular del descenso de gradiente.

- El optimizador tiene parámetros de configuración, siendo el más importante la **tasa de aprendizaje** (`lr`).

- Los optimizadores necesitan saber qué parámetros optimizar, por lo que se pasa `model.parameters()`.

In [7]:
loss_fn = nn.BCELoss()  # binary cross-entropy
optimizer = optim.Adam(model.parameters(), lr=0.001)

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section4"></a>
# <font color="#004D7F" size=6>4. Entrenamiento del Modelo</font>

- Una vez definidos el modelo, la métrica de pérdida y el optimizador, el modelo está listo para ser entrenado con datos.

- El entrenamiento de una red neuronal se realiza generalmente en **épocas** y **lotes**.

    - Una **época** implica pasar el conjunto de datos completo al modelo una vez.

    - Un **lote** es un conjunto de muestras que se pasan al modelo para una iteración del algoritmo de descenso de gradiente.

- El conjunto de datos se divide en lotes que se pasan al modelo secuencialmente en un bucle de entrenamiento.

- Completar todos los lotes constituye una época.

- El proceso se repite para múltiples épocas hasta lograr un rendimiento satisfactorio del modelo.

- El tamaño del lote está limitado por la memoria del sistema y afecta la cantidad de cómputos necesarios.

- El número total de lotes durante varias épocas determina cuántas veces se ejecuta el descenso de gradiente.

- El número de épocas y el tamaño del lote se ajustan mediante prueba y error.

- El objetivo del entrenamiento es que el modelo aprenda la relación entre datos de entrada y salidas esperadas.

- Es normal tener errores, pero estos generalmente disminuyen en épocas posteriores hasta estabilizarse (convergencia del modelo).

- La forma más simple de construir un bucle de entrenamiento es usar dos bucles anidados: uno para épocas y otro para lotes.

In [8]:
n_epochs = 100
batch_size = 10

for epoch in range(n_epochs):
    for i in range(0, len(X), batch_size):
        Xbatch = X[i:i+batch_size]
        ybatch = y[i:i+batch_size]
        y_pred = model(Xbatch)
        loss = loss_fn(y_pred, ybatch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f'Finished epoch {epoch}, latest loss {loss}')

Finished epoch 0, latest loss 0.5566398501396179
Finished epoch 1, latest loss 0.5184499025344849
Finished epoch 2, latest loss 0.5256586074829102
Finished epoch 3, latest loss 0.5243189930915833
Finished epoch 4, latest loss 0.5215479135513306
Finished epoch 5, latest loss 0.533611536026001


Finished epoch 6, latest loss 0.5400281548500061
Finished epoch 7, latest loss 0.546938955783844
Finished epoch 8, latest loss 0.5527857542037964
Finished epoch 9, latest loss 0.555712878704071
Finished epoch 10, latest loss 0.5617987513542175
Finished epoch 11, latest loss 0.5606862902641296


Finished epoch 12, latest loss 0.5637898445129395
Finished epoch 13, latest loss 0.5668650269508362
Finished epoch 14, latest loss 0.5674461126327515
Finished epoch 15, latest loss 0.5693966150283813
Finished epoch 16, latest loss 0.5668600797653198
Finished epoch 17, latest loss 0.5645343661308289


Finished epoch 18, latest loss 0.5746724605560303
Finished epoch 19, latest loss 0.5752333998680115
Finished epoch 20, latest loss 0.5731880068778992
Finished epoch 21, latest loss 0.5743480324745178
Finished epoch 22, latest loss 0.5687615871429443
Finished epoch 23, latest loss 0.5703517198562622


Finished epoch 24, latest loss 0.5792745351791382
Finished epoch 25, latest loss 0.5706915855407715
Finished epoch 26, latest loss 0.5650545358657837
Finished epoch 27, latest loss 0.5670205950737
Finished epoch 28, latest loss 0.5607448816299438
Finished epoch 29, latest loss 0.5595560073852539


...

Finished epoch 72, latest loss 0.4939076006412506
Finished epoch 73, latest loss 0.48498526215553284
Finished epoch 74, latest loss 0.48526865243911743
Finished epoch 75, latest loss 0.4843336045742035
Finished epoch 76, latest loss 0.4816301763057709
Finished epoch 77, latest loss 0.4816291034221649


Finished epoch 78, latest loss 0.4758974015712738
Finished epoch 79, latest loss 0.4773007333278656
Finished epoch 80, latest loss 0.481593519449234
Finished epoch 81, latest loss 0.47509148716926575
Finished epoch 82, latest loss 0.47808578610420227
Finished epoch 83, latest loss 0.47627589106559753


Finished epoch 84, latest loss 0.470054030418396
Finished epoch 85, latest loss 0.46663492918014526
Finished epoch 86, latest loss 0.46447351574897766
Finished epoch 87, latest loss 0.46513038873672485
Finished epoch 88, latest loss 0.4633163511753082
Finished epoch 89, latest loss 0.4645726978778839


Finished epoch 90, latest loss 0.46570608019828796
Finished epoch 91, latest loss 0.4649568200111389
Finished epoch 92, latest loss 0.4618949890136719
Finished epoch 93, latest loss 0.463943749666214
Finished epoch 94, latest loss 0.45976096391677856
Finished epoch 95, latest loss 0.4624079763889313


Finished epoch 96, latest loss 0.4620566666126251
Finished epoch 97, latest loss 0.46608901023864746
Finished epoch 98, latest loss 0.46286481618881226
Finished epoch 99, latest loss 0.4619160592556


### Notas del código:
Este bloque de código implementa el bucle de entrenamiento de un modelo en PyTorch utilizando un enfoque de procesamiento por lotes (batch processing). Vamos a desglosarlo paso a paso para entender qué hace:

1. Definición de hiperparámetros
```python
		n_epochs = 100
		batch_size = 10
```
- `n_epochs`: Es el número total de épocas, es decir, cuántas veces se recorrerán todos los datos de entrenamiento.
- `batch_size`: Es el tamaño del lote (batch), es decir, cuántos ejemplos de datos se procesan a la vez antes de actualizar los parámetros del modelo.

2. Bucle externo: Épocas
```python
		for epoch in range(n_epochs):
```
- El bucle externo controla el número de épocas. En cada iteración, se recorrerán todos los datos de entrenamiento una vez.

3. Bucle interno: Mini-lotes
```python
		for i in range(0, len(X), batch_size):
```
- Este bucle recorre los datos de entrenamiento en fragmentos de tamaño batch_size desde el índice i hasta i + batch_size.
- `X[i:i+batch_size]`: Extrae un sublote (o mini-lote) de datos de entrada.
- `y[i:i+batch_size]`: Extrae el mini-lote correspondiente de etiquetas.

4. Predicción del modelo
```python
		y_pred = model(Xbatch)
```
- Calcula las predicciones del modelo para el mini-lote de entrada Xbatch.

5. Cálculo de la pérdida
```python
		loss = loss_fn(y_pred, ybatch)
```
- `loss_fn`: Es la función de pérdida que compara las predicciones y_pred del modelo con las etiquetas reales ybatch.
- `loss`: Es el valor de la pérdida para el mini-lote actual.

6. Propagación hacia atrás
```python
		optimizer.zero_grad()
		loss.backward()
		optimizer.step()
```
- `optimizer.zero_grad()`: Limpia los gradientes acumulados de las iteraciones anteriores.
- `loss.backward()`: Calcula los gradientes de la pérdida respecto a los parámetros del modelo usando backpropagation.
- `optimizer.step()`: Actualiza los parámetros del modelo basándose en los gradientes calculados y la estrategia de optimización definida en optimizer.

7. Impresión de estado
```python
		print(f'Finished epoch {epoch}, latest loss {loss}')
```

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section5"></a>
# <font color="#004D7F" size=6>5. Evaluar el modelo</font>

- Puedes evaluar el rendimiento de la red en el mismo conjunto de datos utilizado para el entrenamiento.

- Esta evaluación solo proporciona una idea de qué tan bien se ha modelado el conjunto de datos (por ejemplo, la precisión de entrenamiento).

- Sin embargo, no indica cómo podría desempeñarse el algoritmo con datos nuevos.

- Este enfoque se utilizó por simplicidad, pero lo ideal es separar los datos en conjuntos de entrenamiento y validación.

- Puedes evaluar el modelo en el conjunto de entrenamiento de la misma manera que lo invocaste durante el entrenamiento.

- Esto generará predicciones para cada entrada.

- Esta puntuación puede ser la misma que tu función de pérdida o algo diferente.

- Para una clasificación binaria, puedes usar accuracy como puntuación de evaluación.

In [9]:
with torch.no_grad():
    y_pred = model(X)
accuracy = (y_pred.round() == y).float().mean()

- La función `round()` redondea el número decimal al entero más cercano.

- El operador `==` compara y devuelve un tensor booleano, que puede convertirse en números decimales 1.0 y 0.0.

- La función `mean()` proporciona el conteo de 1's (predicciones que coinciden con la etiqueta) dividido por el total de muestras.

- El contexto `no_grad()` es opcional pero recomendado para liberar a `y_pred` de recordar cómo se obtuvo el número.

- Idealmente, la pérdida debería acercarse a cero y el accuracy a 1.0 (100%).



In [10]:
print(f"Accuracy {accuracy}")

Accuracy 0.7591145634651184


- En problemas de aprendizaje automático no triviales, siempre habrá algún error en el modelo.

- El objetivo es elegir una configuración de modelo y entrenamiento que logre la menor pérdida y la mayor precisión posible para un conjunto de datos dado.


---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section6"></a>
# <font color="#004D7F" size=6>6. Hacer predicciones</font>

- Se puede adaptar el ejemplo anterior para generar predicciones en el conjunto de datos de entrenamiento.

- Se trata el conjunto de entrenamiento como si fuera un nuevo conjunto de datos no visto.

- Hacer predicciones es tan simple como llamar al modelo como si fuera una función.

- La función de activación sigmoide en la capa de salida produce predicciones en forma de probabilidades entre 0 y 1.

- Estas probabilidades se pueden convertir fácilmente en predicciones binarias claras mediante redondeo.

In [11]:
with torch.no_grad():
    predictions = model(X)
for i in range(5):
    print(f"{X[i].tolist()} => {predictions[i].item():.4f} (esperado {y[i].item()})")

[6.0, 148.0, 72.0, 35.0, 0.0, 33.599998474121094, 0.6269999742507935, 50.0] => 0.6735 (esperado 1.0)
[1.0, 85.0, 66.0, 29.0, 0.0, 26.600000381469727, 0.35100001096725464, 31.0] => 0.1105 (esperado 0.0)
[8.0, 183.0, 64.0, 0.0, 0.0, 23.299999237060547, 0.671999990940094, 32.0] => 0.8200 (esperado 1.0)
[1.0, 89.0, 66.0, 23.0, 94.0, 28.100000381469727, 0.16699999570846558, 21.0] => 0.1092 (esperado 0.0)
[0.0, 137.0, 40.0, 35.0, 168.0, 43.099998474121094, 2.2880001068115234, 33.0] => 0.5571 (esperado 1.0)


Alternativamente, puedes convertir la probabilidad en 0 o 1 para predecir clases claras directamente; por ejemplo:

In [12]:
with torch.no_grad():
    predictions = model(X).round()  # probabilidad -> clase 0/1
for i in range(5):
    print(f"{X[i].tolist()} => {int(predictions[i].item())} (esperado {int(y[i].item())})")

[6.0, 148.0, 72.0, 35.0, 0.0, 33.599998474121094, 0.6269999742507935, 50.0] => 1 (esperado 1)
[1.0, 85.0, 66.0, 29.0, 0.0, 26.600000381469727, 0.35100001096725464, 31.0] => 0 (esperado 0)
[8.0, 183.0, 64.0, 0.0, 0.0, 23.299999237060547, 0.671999990940094, 32.0] => 1 (esperado 1)
[1.0, 89.0, 66.0, 23.0, 94.0, 28.100000381469727, 0.16699999570846558, 21.0] => 0 (esperado 0)
[0.0, 137.0, 40.0, 35.0, 168.0, 43.099998474121094, 2.2880001068115234, 33.0] => 1 (esperado 1)


<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#004D7F"></i> </font></div>